In [1]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "RegulatoryRAG/0.1 (academic project)"
}

def fetch_page(url, timeout=30):
    """Download one web page and return its HTML as text."""
    response = requests.get(url, headers=HEADERS, timeout=timeout)
    response.raise_for_status()
    return response.text


CBN_BSD = "https://www.cbn.gov.ng/Documents/bsdcirculars.html"

html = fetch_page(CBN_BSD)
print("Characters received:", len(html))

Characters received: 7356


In [2]:
def find_pdf_links(html, base_url):
    """Return every PDF link on the page as {'title': ..., 'url': ...}."""
    soup = BeautifulSoup(html, "lxml")
    links = []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if ".pdf" in href.lower():
            links.append({
                "title": " ".join(a.get_text().split()),
                "url": urljoin(base_url, href),
            })
    return links


pdfs = find_pdf_links(html, CBN_BSD)

print("PDF links found:", len(pdfs))
for p in pdfs[:8]:
    print("-", p["title"][:70], "|", p["url"])

PDF links found: 0


In [3]:
CBN_SUPERVISION = "https://www.cbn.gov.ng/api/GetSupervisionCirculars?format=json"
CBN_ALL = "https://www.cbn.gov.ng/api/GetAllCirculars?format=json"


def fetch_json(url, timeout=60):
    """Download a JSON endpoint and return it as Python data."""
    response = requests.get(url, headers=HEADERS, timeout=timeout)
    response.raise_for_status()
    return response.json()


supervision = fetch_json(CBN_SUPERVISION)

print("Records returned:", len(supervision))
print()
for key, value in supervision[0].items():
    print(f"{key}: {str(value)[:80]}")

Records returned: 556

id: 8126
clickCount: 526
refNo: FPR/DIR/PUB/CIR/001/017
title: Exposure Draft of The Revised Guidelines for Licensing and Regulating Financial 
description: 
author: 
keywords: Exposure Draft
link: /Out/2026/CCD/Holdco circular and guidelines signed.pdf
documentDate: 11/06/2026
filesize: 1032953


In [4]:
from urllib.parse import quote

BASE = "https://www.cbn.gov.ng"


def normalise_cbn(records, regulator="CBN"):
    """Convert CBN API records into our common document shape."""
    documents = []
    for r in records:
        link = (r.get("link") or "").strip()
        if not link.lower().endswith(".pdf"):
            continue
        documents.append({
            "regulator": regulator,
            "ref_no": (r.get("refNo") or "").strip(),
            "title": " ".join((r.get("title") or "").split()),
            "url": BASE + quote(link),
            "date": (r.get("documentDate") or "").strip(),
        })
    return documents


cbn_docs = normalise_cbn(supervision)

print("Usable PDF documents:", len(cbn_docs))
print()
for d in cbn_docs[:5]:
    print(d["date"], "|", d["ref_no"], "|", d["title"][:60])
    print("   ", d["url"])

Usable PDF documents: 552

11/06/2026 | FPR/DIR/PUB/CIR/001/017 | Exposure Draft of The Revised Guidelines for Licensing and R
    https://www.cbn.gov.ng/Out/2026/CCD/Holdco%20circular%20and%20guidelines%20signed.pdf
11/06/2026 | FPR/DIR/PUB/CIR/001/016 | Exposure of The Draft Guidelines on Ring-Fencing Operations 
    https://www.cbn.gov.ng/Out/2026/CCD/THE%20EXPOSURE%20DRAFT%20GUIDELINES%20ON%20RING-FENCING%20OPERATIONS%20OF%20CLOSELY%20LINKED%20ENTITIES.pdf
12/03/2026 | FPR/DIR/PUB/CIR/001/014 | RE: GUIDELINES ON MANAGEMENT OF DORMANT ACCOUNTS, UNCLAIMED 
    https://www.cbn.gov.ng/Out/2026/FPRD/DORMANT%20CIRCULAR.pdf
10/03/2026 | BSD/DIR/PUB/LAB/019/002 | Issuance of Baseline Standards for Automated Anti-Money Laun
    https://www.cbn.gov.ng/Out/2026/CCD/CBN%20issues%20Baseline%20Standards%20for%20Automated%20Anti-Money%20Laundering%20Solution.pdf
18/12/2025 | FPR/DIR/PUB/CIR/01/012 | Circular to all Banks and Other Financial Institutions Facil
    https://www.cbn.gov.ng/Out/2025/C

In [5]:
import pandas as pd

df = pd.DataFrame(cbn_docs)
df["date_parsed"] = pd.to_datetime(df["date"], format="%d/%m/%Y", errors="coerce")
df = df.sort_values("date_parsed", ascending=False).reset_index(drop=True)

print("Total documents:", len(df))
print("Oldest:", df["date_parsed"].min().date())
print("Newest:", df["date_parsed"].max().date())
print("Rows with unreadable dates:", int(df["date_parsed"].isna().sum()))

Total documents: 552
Oldest: 2004-04-23
Newest: 2026-06-11
Rows with unreadable dates: 0


In [8]:
TOPICS = {
    "Capital & recapitalisation": ["capital", "recapitali"],
    "AML / CFT": ["money laundering", "terrorism financing", "aml", "cft"],
    "Corporate governance": ["corporate governance", "board of directors", "whistle"],
    "Licensing": ["licence", "license", "licensing"],
    "Cybersecurity & IT risk": ["cybersecurity", "cyber", "information security"],
    "KYC & customer due diligence": ["know your customer", "customer due diligence", "bvn", "tier"],
    "Consumer protection": ["consumer protection", "dormant account", "complaint", "disclosure"],
    "Prudential & risk": ["prudential", "risk management", "liquidity", "credit risk"],
}

for topic, terms in TOPICS.items():
    pattern = "|".join(terms)
    hits = df[df["title"].str.contains(pattern, case=False, na=False, regex=True)]
    print(f"\n=== {topic}  ({len(hits)} matches) ===")
    for i, row in hits.head(4).iterrows():
        date = row["date_parsed"].date() if pd.notna(row["date_parsed"]) else "no date"
        print(f"[{i}] {date} | {row['ref_no']}")
        print(f"     {row['title'][:95]}")


=== Capital & recapitalisation  (31 matches) ===
[24] 2024-03-28 | FPR/DIR/PUB/CIR/002/009
     Review of Minimum Capital Requirements for Commercial, Merchant and Non-Interest Banks in Niger
[71] 2021-09-02 | BSD/DIR/PUB/14/063
     Revised Guidelines on Supervisory Review Process of Internal Capital Adequacy Assessment Proces
[73] 2021-09-02 | BSD/DIR/PUB/14/063
     Guidelines on Regulatory Capital
[117] 2020-04-29 | FPR/DIR/GEN/CIR/07/054
     Review of Minimum Capital Requirements for Microfinance Banks in Nigeria

=== AML / CFT  (18 matches) ===
[3] 2026-03-10 | BSD/DIR/PUB/LAB/019/002
     Issuance of Baseline Standards for Automated Anti-Money Laundering (AML) Solution for Financial
[4] 2025-12-18 | FPR/DIR/PUB/CIR/01/012
     Circular to all Banks and Other Financial Institutions Facilitation of Seamless Use Of Foreign 
[10] 2025-05-21 | BSD/DIR/CON/AML/018/033
     Exposure of Draft Baseline Standards for Automated Anti-Money Laundering (AML) Solutions
[17] 2024-06-30 | FPR/

In [9]:
CBN_KEEP = [24, 73, 72, 35, 18, 3, 45, 33, 2, 25]

selected = df.loc[CBN_KEEP].copy()

print("Selected:", len(selected))
for _, r in selected.iterrows():
    print(r["date_parsed"].date(), "|", r["ref_no"], "|", r["title"][:70])

Selected: 10
2024-03-28 | FPR/DIR/PUB/CIR/002/009 | Review of Minimum Capital Requirements for Commercial, Merchant and No
2021-09-02 | BSD/DIR/PUB/14/063 | Guidelines on Regulatory Capital
2021-09-02 | BSD/DIR/PUB/14/063 | Guidelines on Liquidity Risk Management and Internal Liquidity Adequac
2023-07-14 | FPR/DIR/PUB/CIR/001/078 | Corporate Governance Guidelines for Commercial Banks, Merchant, Non-In
2024-05-31 | BSD/DIR/PUB/LAB/017/008 | Central Bank of Nigeria Risk-Based Cybersecurity Framework and Guideli
2026-03-10 | BSD/DIR/PUB/LAB/019/002 | Issuance of Baseline Standards for Automated Anti-Money Laundering (AM
2022-11-23 | FPR/DIR/GEN/CIR/001/061 | Guidelines for Licensing of Banks and Other Financial Institutions in 
2023-12-08 | FPR/DIR/PUB/CIR/002/002 | Additional Know Your Customer Requirement in Respect of Non-Profit Org
2026-03-12 | FPR/DIR/PUB/CIR/001/014 | RE: GUIDELINES ON MANAGEMENT OF DORMANT ACCOUNTS, UNCLAIMED BALANCES A
2024-03-14 | BSD/DIR/PUB/LAB/017/003 | Re: Im

In [10]:
from pathlib import Path
import re

RAW_DIR = Path("../data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)


def make_filename(regulator, ref_no, title, max_len=90):
    """Build a safe, descriptive filename from the document's metadata."""
    base = f"{regulator}_{ref_no}_{title}"
    base = re.sub(r"[^A-Za-z0-9]+", "_", base).strip("_")
    return base[:max_len] + ".pdf"


def download_pdf(url, destination, timeout=120):
    """Download one PDF. Returns False if it was already on disk."""
    if destination.exists():
        return False

    response = requests.get(url, headers=HEADERS, timeout=timeout)
    response.raise_for_status()

    if not response.content.startswith(b"%PDF"):
        raise ValueError(
            f"Not a PDF. Content-Type was {response.headers.get('Content-Type')}, "
            f"first bytes were {response.content[:20]!r}"
        )

    destination.write_bytes(response.content)
    return True


row = df.loc[24]
destination = RAW_DIR / make_filename(row["regulator"], row["ref_no"], row["title"])

saved = download_pdf(row["url"], destination)

print("Newly downloaded:", saved)
print("Filename:", destination.name)
print("Size:", round(destination.stat().st_size / 1024, 1), "KB")

HTTPError: 403 Client Error: Forbidden for url: https://www.cbn.gov.ng/Out/2024/CCD/Recapitalization_MARCH_2024.pdf

In [11]:
test_url = df.loc[24, "url"]

BROWSER_UA = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
)
LISTING_PAGE = "https://www.cbn.gov.ng/Documents/bsdcirculars.html"

# A - exactly what failed
r = requests.get(test_url, headers=HEADERS, timeout=60)
print("A  original headers        ->", r.status_code)

# B - browser user-agent only
r = requests.get(test_url, headers={"User-Agent": BROWSER_UA}, timeout=60)
print("B  browser user-agent      ->", r.status_code)

# C - browser user-agent plus referer
r = requests.get(
    test_url,
    headers={"User-Agent": BROWSER_UA, "Referer": LISTING_PAGE},
    timeout=60,
)
print("C  user-agent + referer    ->", r.status_code)

# D - a session that visits the listing page first, collecting cookies
s = requests.Session()
s.headers.update({
    "User-Agent": BROWSER_UA,
    "Accept": "application/pdf,text/html;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": LISTING_PAGE,
})
s.get(LISTING_PAGE, timeout=60)
r = s.get(test_url, timeout=60)
print("D  warmed session          ->", r.status_code,
      "| real PDF:", r.content.startswith(b"%PDF"),
      "| bytes:", len(r.content))

A  original headers        -> 403
B  browser user-agent      -> 403
C  user-agent + referer    -> 403
D  warmed session          -> 403 | real PDF: False | bytes: 5946


In [12]:
def normalise_cbn(records, regulator="CBN"):
    """Convert CBN API records into our common document shape."""
    documents = []
    for r in records:
        link = (r.get("link") or "").strip()
        if not link.lower().endswith(".pdf"):
            continue
        documents.append({
            "regulator": regulator,
            "ref_no": (r.get("refNo") or "").strip(),
            "title": " ".join((r.get("title") or "").split()),
            "url": BASE + quote(link.upper()),   # <- the fix
            "date": (r.get("documentDate") or "").strip(),
        })
    return documents


cbn_docs = normalise_cbn(supervision)

df = pd.DataFrame(cbn_docs)
df["date_parsed"] = pd.to_datetime(df["date"], format="%d/%m/%Y", errors="coerce")
df = df.sort_values("date_parsed", ascending=False).reset_index(drop=True)

print(df.loc[24, "url"])

https://www.cbn.gov.ng/OUT/2024/CCD/RECAPITALIZATION_MARCH_2024.PDF


In [13]:
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": BROWSER_UA, "Referer": LISTING_PAGE})


def download_pdf(url, destination, timeout=120):
    """Download one PDF. Returns False if it was already on disk."""
    if destination.exists():
        return False

    response = SESSION.get(url, timeout=timeout)
    response.raise_for_status()

    if not response.content.startswith(b"%PDF"):
        raise ValueError(
            f"Not a PDF. Content-Type was {response.headers.get('Content-Type')}, "
            f"first bytes were {response.content[:20]!r}"
        )

    destination.write_bytes(response.content)
    return True


row = df.loc[24]
destination = RAW_DIR / make_filename(row["regulator"], row["ref_no"], row["title"])

saved = download_pdf(row["url"], destination)

print("Newly downloaded:", saved)
print("Filename:", destination.name)
print("Size:", round(destination.stat().st_size / 1024, 1), "KB")

Newly downloaded: True
Filename: CBN_FPR_DIR_PUB_CIR_002_009_Review_of_Minimum_Capital_Requirements_for_Commercial_Merchant.pdf
Size: 221.2 KB


In [14]:
import time

selected = df.loc[CBN_KEEP].copy()   # df was rebuilt, so refresh the selection

records = []

for idx, row in selected.iterrows():
    filename = make_filename(row["regulator"], row["ref_no"], row["title"])
    destination = RAW_DIR / filename

    try:
        saved = download_pdf(row["url"], destination)
        status = "downloaded" if saved else "already present"
        error = ""
    except Exception as exc:
        status = "FAILED"
        error = f"{type(exc).__name__}: {exc}"[:150]

    size_kb = round(destination.stat().st_size / 1024, 1) if destination.exists() else 0

    records.append({
        "filename": filename,
        "regulator": row["regulator"],
        "ref_no": row["ref_no"],
        "title": row["title"],
        "url": row["url"],
        "document_date": row["date"],
        "size_kb": size_kb,
        "status": status,
        "error": error,
        "downloaded_on": pd.Timestamp.today().date().isoformat(),
    })

    print(f"{status:<16} {size_kb:>8} KB  {filename[:55]}")
    time.sleep(2)

sources = pd.DataFrame(records)
sources.to_csv("../data/sources.csv", index=False)

print("\n--- summary ---")
print(sources["status"].value_counts().to_string())
print("Total:", round(sources['size_kb'].sum() / 1024, 2), "MB")
print("\nFailures:")
print(sources.loc[sources["status"] == "FAILED", ["ref_no", "error"]].to_string(index=False))

already present     221.2 KB  CBN_FPR_DIR_PUB_CIR_002_009_Review_of_Minimum_Capital_R
downloaded          537.8 KB  CBN_BSD_DIR_PUB_14_063_Guidelines_on_Regulatory_Capital
downloaded          268.8 KB  CBN_BSD_DIR_PUB_14_063_Guidelines_on_Liquidity_Risk_Man
downloaded         1309.9 KB  CBN_FPR_DIR_PUB_CIR_001_078_Corporate_Governance_Guidel
downloaded         1030.0 KB  CBN_BSD_DIR_PUB_LAB_017_008_Central_Bank_of_Nigeria_Ris
downloaded          411.8 KB  CBN_BSD_DIR_PUB_LAB_019_002_Issuance_of_Baseline_Standa
downloaded          490.6 KB  CBN_FPR_DIR_GEN_CIR_001_061_Guidelines_for_Licensing_of
downloaded          211.4 KB  CBN_FPR_DIR_PUB_CIR_002_002_Additional_Know_Your_Custom
downloaded          728.7 KB  CBN_FPR_DIR_PUB_CIR_001_014_RE_GUIDELINES_ON_MANAGEMENT
downloaded          187.1 KB  CBN_BSD_DIR_PUB_LAB_017_003_Re_Impact_of_Recent_Policy_

--- summary ---
status
downloaded         9
already present    1
Total: 5.27 MB

Failures:
Empty DataFrame
Columns: [ref_no, error]
Index: 

In [15]:
from urllib.parse import urlparse

def fetch_page(url, timeout=60):
    """Download one web page and return its HTML as text."""
    response = requests.get(url, headers={"User-Agent": BROWSER_UA}, timeout=timeout)
    response.raise_for_status()
    return response.text


def download_pdf(url, destination, timeout=180):
    """Download one PDF. Returns False if it was already on disk."""
    if destination.exists():
        return False

    parsed = urlparse(url)
    headers = {
        "User-Agent": BROWSER_UA,
        "Referer": f"{parsed.scheme}://{parsed.netloc}/",
    }
    response = requests.get(url, headers=headers, timeout=timeout)
    response.raise_for_status()

    if not response.content.startswith(b"%PDF"):
        raise ValueError(
            f"Not a PDF. Content-Type was {response.headers.get('Content-Type')}, "
            f"first bytes were {response.content[:20]!r}"
        )

    destination.write_bytes(response.content)
    return True

In [16]:
def scrape_pdf_page(page_url, regulator):
    """Find every PDF linked on an ordinary HTML page."""
    links = find_pdf_links(fetch_page(page_url), page_url)
    seen, docs = set(), []
    for l in links:
        if l["url"] in seen:
            continue
        seen.add(l["url"])
        title = l["title"] or Path(urlparse(l["url"]).path).stem.replace("_", " ").replace("-", " ")
        docs.append({
            "regulator": regulator,
            "ref_no": "",
            "title": " ".join(title.split())[:150],
            "url": l["url"],
            "date": "",
            "source_page": page_url,
        })
    return docs


PAGES = [
    ("https://ndic.gov.ng/supervision/supervisory-guidelines-standards/", "NDIC"),
    ("https://ndic.gov.ng/resources/publications/", "NDIC"),
    ("https://home.sec.gov.ng/our-mandate/regulation/rules-and-regulations/", "SEC"),
]

other_docs = []
for page_url, regulator in PAGES:
    try:
        found = scrape_pdf_page(page_url, regulator)
        print(f"{regulator:<5} {len(found):>3} PDFs   {page_url}")
        other_docs.extend(found)
    except Exception as exc:
        print(f"{regulator:<5} FAILED  {type(exc).__name__}: {exc}")

other = pd.DataFrame(other_docs).drop_duplicates(subset="url").reset_index(drop=True)

print("\nTotal unique:", len(other))
print()
for i, r in other.iterrows():
    print(f"[{i}] {r['regulator']:<5} | {r['title'][:85]}")

NDIC    5 PDFs   https://ndic.gov.ng/supervision/supervisory-guidelines-standards/
NDIC  FAILED  ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
SEC   FAILED  ConnectionError: HTTPSConnectionPool(host='home.sec.gov.ng', port=443): Max retries exceeded with url: /our-mandate/regulation/rules-and-regulations/ (Caused by NameResolutionError("HTTPSConnection(host='home.sec.gov.ng', port=443): Failed to resolve 'home.sec.gov.ng' ([Errno 11002] getaddrinfo failed)"))

Total unique: 5

[0] NDIC  | Banking supervision
[1] NDIC  | Types of Bank Supervision
[2] NDIC  | Supervisory Guidelines
[3] NDIC  | SUPERVISORY ACTIVITIES
[4] NDIC  | FOCUS OF SUPERVISORY ACTIVITIES


In [17]:
import time

def fetch_page(url, timeout=60, retries=3):
    """Download a page, retrying transient network failures."""
    last_error = None
    for attempt in range(1, retries + 1):
        try:
            response = requests.get(url, headers={"User-Agent": BROWSER_UA}, timeout=timeout)
            response.raise_for_status()
            return response.text
        except requests.exceptions.RequestException as exc:
            last_error = exc
            if attempt < retries:
                wait = 2 ** attempt
                print(f"    attempt {attempt} failed ({type(exc).__name__}), retrying in {wait}s")
                time.sleep(wait)
    raise last_error


PAGES = [
    ("https://ndic.gov.ng/supervision/supervisory-guidelines-standards/", "NDIC"),
    ("https://ndic.gov.ng/resources/publications/", "NDIC"),
    ("https://sec.gov.ng/our-mandate/regulation/rules-and-regulations/", "SEC"),
]

other_docs = []
for page_url, regulator in PAGES:
    try:
        found = scrape_pdf_page(page_url, regulator)
        print(f"{regulator:<5} {len(found):>3} PDFs   {page_url}")
        other_docs.extend(found)
    except Exception as exc:
        print(f"{regulator:<5} FAILED  {type(exc).__name__}: {str(exc)[:90]}")

other = pd.DataFrame(other_docs).drop_duplicates(subset="url").reset_index(drop=True)

print("\nTotal unique:", len(other))
for i, r in other.iterrows():
    print(f"\n[{i}] {r['regulator']} | {r['title'][:80]}")
    print(f"     {r['url']}")

NDIC    5 PDFs   https://ndic.gov.ng/supervision/supervisory-guidelines-standards/
NDIC    5 PDFs   https://ndic.gov.ng/resources/publications/
SEC    30 PDFs   https://sec.gov.ng/our-mandate/regulation/rules-and-regulations/

Total unique: 40

[0] NDIC | Banking supervision
     https://ndic.gov.ng/storage/cms/documents/banking-supervision-cbb5dd67-9fad-48ac-9989-1d224ba53443.pdf

[1] NDIC | Types of Bank Supervision
     https://ndic.gov.ng/storage/cms/documents/types-of-bank-supervision-6f132d19-b40b-4f9e-a20c-5c376d16e9f5.pdf

[2] NDIC | Supervisory Guidelines
     https://ndic.gov.ng/storage/cms/documents/supervisory-guidelines-3083de20-a363-472c-92ce-2774b1ebebfc.pdf

[3] NDIC | SUPERVISORY ACTIVITIES
     https://ndic.gov.ng/storage/cms/documents/supervisory-activities-3226d5c7-74d1-4642-9cfd-e8798f24b9a6.pdf

[4] NDIC | FOCUS OF SUPERVISORY ACTIVITIES
     https://ndic.gov.ng/storage/cms/documents/focus-of-supervisory-activities-1d03653e-f817-4de8-8ae6-8f108816f0c3.pdf

[5] NDI

In [18]:
SEED_DOCS = [
    {"regulator": "NDIC", "ref_no": "NDIC Act 2023", "date": "",
     "title": "Nigeria Deposit Insurance Corporation Act 2023",
     "url": "https://ndic.gov.ng/wp-content/uploads/2023/09/NDIC-Act-2023-LATEST.pdf"},
    {"regulator": "SEC", "ref_no": "SEC Rules 2019", "date": "",
     "title": "SEC New Rules and Amendments, December 2019",
     "url": "https://sec.gov.ng/wp-content/uploads/2019/12/SEC-New-Rules-and-Ammendments-23-December-2019.pdf"},
]

In [19]:
NDIC_KEEP = [9, 2]
SEC_KEEP  = [16, 33, 14, 15, 17, 38]

extra = other.loc[NDIC_KEEP + SEC_KEEP].copy()

LEGISLATION = pd.DataFrame([
    {"regulator": "CBN", "ref_no": "BOFIA 2020", "date": "",
     "title": "Banks and Other Financial Institutions Act 2020",
     "url": "https://www.cbn.gov.ng/OUT/2021/CCD/BOFIA%202020.PDF"},
    {"regulator": "FGN", "ref_no": "MLPPA 2022", "date": "",
     "title": "Money Laundering Prevention and Prohibition Act 2022",
     "url": "https://placng.org/i/wp-content/uploads/2022/05/Money-Laundering-Prevention-and-Prohibition-Act-2022.pdf"},
])

to_download = pd.concat([extra, LEGISLATION], ignore_index=True)


def download_many(frame, pause=2.0):
    """Download every row, recording what happened to each."""
    records = []
    for _, row in frame.iterrows():
        filename = make_filename(row["regulator"], row["ref_no"] or "NA", row["title"])
        destination = RAW_DIR / filename

        try:
            saved = download_pdf(row["url"], destination)
            status = "downloaded" if saved else "already present"
            error = ""
        except Exception as exc:
            status = "FAILED"
            error = f"{type(exc).__name__}: {exc}"[:150]

        size_kb = round(destination.stat().st_size / 1024, 1) if destination.exists() else 0
        if status != "FAILED" and size_kb < 30:
            status = "CHECK - tiny"

        records.append({
            "filename": filename, "regulator": row["regulator"], "ref_no": row["ref_no"],
            "title": row["title"], "url": row["url"], "document_date": row.get("date", ""),
            "size_kb": size_kb, "status": status, "error": error,
            "downloaded_on": pd.Timestamp.today().date().isoformat(),
        })
        print(f"{status:<16}{size_kb:>9} KB  {filename[:52]}")
        time.sleep(pause)

    return pd.DataFrame(records)


new_sources = download_many(to_download)

sources = (pd.concat([sources, new_sources], ignore_index=True)
             .drop_duplicates(subset="filename", keep="last")
             .reset_index(drop=True))
sources.to_csv("../data/sources.csv", index=False)

print("\n--- corpus ---")
print(sources["regulator"].value_counts().to_string())
print("Documents:", len(sources))
print("Total size:", round(sources["size_kb"].sum() / 1024, 2), "MB")
print("\nNeeds attention:")
print(sources.loc[sources["status"].isin(["FAILED", "CHECK - tiny"]),
                  ["ref_no", "size_kb", "status", "error"]].to_string(index=False))

downloaded          672.8 KB  NDIC_NA_Preview_Document.pdf
downloaded          136.9 KB  NDIC_NA_Supervisory_Guidelines.pdf
downloaded          762.7 KB  SEC_NA_Download_Full_Document.pdf
already present     762.7 KB  SEC_NA_Download_full_document.pdf
already present     762.7 KB  SEC_NA_Download_Full_Document.pdf
already present     762.7 KB  SEC_NA_Download_Full_Document.pdf
already present     762.7 KB  SEC_NA_Download_Full_Document.pdf
downloaded          380.3 KB  SEC_NA_Rules_Relating_to_the_Complaints_Management_F
downloaded         2374.8 KB  CBN_BOFIA_2020_Banks_and_Other_Financial_Institution
downloaded          574.1 KB  FGN_MLPPA_2022_Money_Laundering_Prevention_and_Prohi

--- corpus ---
regulator
CBN     11
SEC      3
NDIC     2
FGN      1
Documents: 17
Total size: 10.8 MB

Needs attention:
Empty DataFrame
Columns: [ref_no, size_kb, status, error]
Index: []


In [20]:
import hashlib

def make_filename(regulator, ref_no, title, url, max_len=70):
    """Safe filename that cannot collide, because the URL hash is appended."""
    stem = f"{regulator}_{ref_no or 'NA'}_{title}"
    stem = re.sub(r"[^A-Za-z0-9]+", "_", stem).strip("_")[:max_len]
    digest = hashlib.sha1(url.encode()).hexdigest()[:8]
    return f"{stem}_{digest}.pdf"


GENERIC = re.compile(r"^(download|preview)\b", re.IGNORECASE)

def better_title(title, url):
    """If the link text was a button label, use the filename from the URL instead."""
    if title and not GENERIC.match(title.strip()):
        return title
    stem = Path(urlparse(url).path).stem
    stem = re.sub(r"[_\-]+", " ", stem)
    stem = re.sub(r"\s+[0-9a-f]{6,}$", "", stem)      # drop trailing hashes
    return " ".join(stem.split()).strip()


to_download = to_download.copy()
to_download["title"] = [better_title(t, u) for t, u in zip(to_download["title"], to_download["url"])]

for _, r in to_download.iterrows():
    print(f"{r['regulator']:<5} | {r['title'][:75]}")

NDIC  | ndic act 2023 latest 14ab1a0a 82c7 4e05 9419
NDIC  | Supervisory Guidelines
SEC   | SEC AMLCFTCPF REGULATIONS 12 MAY 2022
SEC   | SEC Consolidated JUNE2013 SIGNEDWEBSITE 1
SEC   | New Rules and sundry amendments April 2025
SEC   | Executed Rules Dec 2024
SEC   | Rules on Issuance Offering and Custody of Digital Assets
SEC   | Rules Relating to the Complaints Management Framework of the Nigerian Capit
CBN   | Banks and Other Financial Institutions Act 2020
FGN   | Money Laundering Prevention and Prohibition Act 2022


In [21]:
for bad in RAW_DIR.glob("*_NA_*.pdf"):
    bad.unlink()
    print("removed", bad.name)

sources = sources[~sources["filename"].str.contains("_NA_")].reset_index(drop=True)

def download_many(frame, pause=2.0):
    records = []
    for _, row in frame.iterrows():
        filename = make_filename(row["regulator"], row["ref_no"], row["title"], row["url"])
        destination = RAW_DIR / filename
        try:
            saved = download_pdf(row["url"], destination)
            status = "downloaded" if saved else "already present"
            error = ""
        except Exception as exc:
            status, error = "FAILED", f"{type(exc).__name__}: {exc}"[:150]

        size_kb = round(destination.stat().st_size / 1024, 1) if destination.exists() else 0
        if status != "FAILED" and size_kb < 30:
            status = "CHECK - tiny"

        records.append({
            "filename": filename, "regulator": row["regulator"], "ref_no": row["ref_no"],
            "title": row["title"], "url": row["url"], "document_date": row.get("date", ""),
            "size_kb": size_kb, "status": status, "error": error,
            "downloaded_on": pd.Timestamp.today().date().isoformat(),
        })
        print(f"{status:<16}{size_kb:>9} KB  {filename[:55]}")
        time.sleep(pause)
    return pd.DataFrame(records)


new_sources = download_many(to_download)

sources = (pd.concat([sources, new_sources], ignore_index=True)
             .drop_duplicates(subset="url", keep="last")
             .reset_index(drop=True))
sources.to_csv("../data/sources.csv", index=False)

print("\n--- corpus ---")
print(sources["regulator"].value_counts().to_string())
print("Documents:", len(sources), " | unique files on disk:", len(list(RAW_DIR.glob("*.pdf"))))
print("Total size:", round(sources["size_kb"].sum() / 1024, 2), "MB")

removed NDIC_NA_Preview_Document.pdf
removed NDIC_NA_Supervisory_Guidelines.pdf
removed SEC_NA_Download_Full_Document.pdf
removed SEC_NA_Rules_Relating_to_the_Complaints_Management_Framework_of_the_Nigerian_Capital_Marke.pdf
downloaded          672.8 KB  NDIC_NA_ndic_act_2023_latest_14ab1a0a_82c7_4e05_9419_36
downloaded          136.9 KB  NDIC_NA_Supervisory_Guidelines_c1891a69.pdf
downloaded          762.7 KB  SEC_NA_SEC_AMLCFTCPF_REGULATIONS_12_MAY_2022_fde31457.p
downloaded         2846.0 KB  SEC_NA_SEC_Consolidated_JUNE2013_SIGNEDWEBSITE_1_0eaf60
downloaded         1059.1 KB  SEC_NA_New_Rules_and_sundry_amendments_April_2025_cdbc5
downloaded          716.3 KB  SEC_NA_Executed_Rules_Dec_2024_df195fed.pdf
downloaded         1098.3 KB  SEC_NA_Rules_on_Issuance_Offering_and_Custody_of_Digita
downloaded          380.3 KB  SEC_NA_Rules_Relating_to_the_Complaints_Management_Fram
downloaded         2374.8 KB  CBN_BOFIA_2020_Banks_and_Other_Financial_Institutions_A
downloaded          574.1

renamed: CBN_FPR_DIR_PUB_CIR_002_009_Review_of_Minimum -> CBN_FPR_DIR_PUB_CIR_002_009_Review_of_Minimum
renamed: CBN_BSD_DIR_PUB_14_063_Guidelines_on_Regulato -> CBN_BSD_DIR_PUB_14_063_Guidelines_on_Regulato
renamed: CBN_BSD_DIR_PUB_14_063_Guidelines_on_Liquidit -> CBN_BSD_DIR_PUB_14_063_Guidelines_on_Liquidit
renamed: CBN_FPR_DIR_PUB_CIR_001_078_Corporate_Governa -> CBN_FPR_DIR_PUB_CIR_001_078_Corporate_Governa
renamed: CBN_BSD_DIR_PUB_LAB_017_008_Central_Bank_of_N -> CBN_BSD_DIR_PUB_LAB_017_008_Central_Bank_of_N
renamed: CBN_BSD_DIR_PUB_LAB_019_002_Issuance_of_Basel -> CBN_BSD_DIR_PUB_LAB_019_002_Issuance_of_Basel
renamed: CBN_FPR_DIR_GEN_CIR_001_061_Guidelines_for_Li -> CBN_FPR_DIR_GEN_CIR_001_061_Guidelines_for_Li
renamed: CBN_FPR_DIR_PUB_CIR_002_002_Additional_Know_Y -> CBN_FPR_DIR_PUB_CIR_002_002_Additional_Know_Y
renamed: CBN_FPR_DIR_PUB_CIR_001_014_RE_GUIDELINES_ON_ -> CBN_FPR_DIR_PUB_CIR_001_014_RE_GUIDELINES_ON_
renamed: CBN_BSD_DIR_PUB_LAB_017_003_Re_Impact_of_Rece -> CBN_BS

In [24]:
# 1. Rename anything still using the old scheme
for i, row in sources.iterrows():
    correct = make_filename(row["regulator"], row["ref_no"], row["title"], row["url"])
    current = row["filename"]
    if correct != current:
        src, dst = RAW_DIR / current, RAW_DIR / correct
        if src.exists() and not dst.exists():
            src.rename(dst)
            print("renamed:", current[:45], "->", correct[:45])
        sources.at[i, "filename"] = correct

# 2. Compare what we expect against what is actually there
expected = set(sources["filename"])
on_disk = {p.name for p in RAW_DIR.glob("*.pdf")}

orphans = sorted(on_disk - expected)
missing = sorted(expected - on_disk)

print("\nOrphans on disk (no record):", len(orphans))
for name in orphans:
    print("   ", name[:70])

print("Missing from disk (recorded but absent):", len(missing))
for name in missing:
    print("   ", name[:70])

# 3. Remove orphans and save
for name in orphans:
    (RAW_DIR / name).unlink()

sources.to_csv("../data/sources.csv", index=False)

print("\n--- final corpus ---")
print(sources["regulator"].value_counts().to_string())
print("Records:", len(sources), "| Files on disk:", len(list(RAW_DIR.glob("*.pdf"))))
print("Total size:", round(sources["size_kb"].sum() / 1024, 2), "MB")


Orphans on disk (no record): 0
Missing from disk (recorded but absent): 0

--- final corpus ---
regulator
CBN     11
SEC      6
NDIC     2
FGN      1
Records: 20 | Files on disk: 20
Total size: 15.64 MB
